In [29]:
import os
import cv2
import os
import numpy as np
import torch
import sys
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '../../')))
import torch
from gestures.network.models.basic_model import BasicModel
from gestures.network.models.super_resolution.safmn import SAFMN
import torch
import time
from gestures.network.models.sr_classifier.SRCnnTinyRadar import CombinedSRDrlnClassifier , MultiSRClassifier , RecSRClass


In [21]:
from fvcore.nn import FlopCountAnalysis
import torchprofile
from ptflops import get_model_complexity_info
def get_model_params(model,model_name ,dummy_input):
    d1 = dummy_input.clone()
    model.eval()
    # Measure the time
    start_time = time.time()
    with torch.no_grad():  # Disable gradient calculation for inference
        _ = model(dummy_input)
    end_time = time.time()

    inference_time = end_time - start_time
    print(model_name)
    print(f"Inference Time: {inference_time:.6f} seconds")
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Total number of parameters: {num_params:,}")


In [23]:
from gestures.network.models.classifiers.tiny_radar import TinyRadarNN


tiny = TinyRadarNN()
dummy_input = torch.randn(5,1,2, 32, 492)
get_model_params(tiny,"TinyRadarNN",dummy_input)


TinyRadarNN
Inference Time: 0.008245 seconds
Total number of parameters: 45,948


In [26]:
from gestures.utils_processing_data import *
from gestures.data_loader2.dataset_factory import *
from gestures.setup import get_pc_cgf

files = os.listdir("/Users/netanelblumenfeld/Downloads/11G/tt")
gestures = [
    "PinchIndex",
    "PinchPinky",
    "FingerSlider",
    "FingerRub",
    "SlowSwipeRL",
    "FastSwipeRL",
    "Push",
    "Pull",
    "PalmTilt",
    "Circle",
    "PalmHold",
    "NoHand",
]
base_dir = "/Users/netanelblumenfeld/Downloads/11G/tt"
task = "sr_classifier"  # task = ["sr", "classifier", "sr_classifier"]
pc, data_dir, output_dir, device = get_pc_cgf()


batch_size = 1
dx, dy = 2,2
pre_processing_funcs = {
    "classifier": torch.nn.Sequential(
        ToTensor(),
        DownSampleOneSample(dx=dx, dy=dy, original_dims=False),
        NormalizeOneSample(),
        DopplerMapOneSample(),
    ),
    "sr_classifier": {
        "hr": torch.nn.Sequential(
            ToTensor(), NormalizeOneSample(), ComplexToRealOneSample()
        ),
        "lr": torch.nn.Sequential(
            ToTensor(),
            DownSampleOneSample(dx=dx, dy=dy, original_dims=False),
            NormalizeOneSample(),
            ComplexToRealOneSample(),
        ),
    },
    "sr": {
        "hr": torch.nn.Sequential(
            ToTensor(), NormalizeOneSample(), ComplexToRealOneSample()
        ),
        "lr": torch.nn.Sequential(
            ToTensor(),
            DownSampleOneSample(dx=dx, dy=dy, original_dims=False),
            NormalizeOneSample(),
            ComplexToRealOneSample(),
        ),
    },

}

data_loader = get_data_loader(
    task, batch_size, gestures, data_dir, pre_processing_funcs[task]
)
for x,y in data_loader['val']:
    lr_imgs = x
    hr_imgs = y
    break



0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

In [27]:
safmn = SAFMN(upscaling_factor=2)
comb = CombinedSRDrlnClassifier(safmn, tiny)
dummy_input = comb.reshape_to_model_output(lr_imgs, hr_imgs, torch.device("cpu"))
get_model_params(comb,"SuperGestNet",dummy_input[0])



SuperGestNet
Inference Time: 0.179208 seconds
Total number of parameters: 272,162


In [28]:
safmn2 = SAFMN(upscaling_factor=2)
safmn3 = SAFMN(upscaling_factor=3)
safmn4 = SAFMN(upscaling_factor=4)
tiny = TinyRadarNN()
multi = MultiSRClassifier(safmn2,safmn3,safmn4,tiny, torch.device("cpu"))
dummy_input = multi.reshape_to_model_output(lr_imgs, hr_imgs, torch.device("cpu"))
get_model_params(multi,"multi",dummy_input[0])


multi
Inference Time: 0.134241 seconds
Total number of parameters: 735,586


In [30]:
safmn = SAFMN(upscaling_factor=2)
rec = RecSRClass(safmn, tiny)
dummy_input = rec.reshape_to_model_output(lr_imgs, hr_imgs, torch.device("cpu"))
get_model_params(rec,"rec",dummy_input[0])



rec
Inference Time: 0.144764 seconds
Total number of parameters: 272,144
